# BirdCLEF 2026 — Extract Perch v2 Embeddings from ALL Train Soundscapes

Extract 1536-dim Perch embeddings + 234-class mapped logits for every 5-sec window in `train_soundscapes/`.

**Outputs** (saved to `/kaggle/working/`):
- `all_ss_meta.parquet` — row_id, filename, site, hour_utc
- `all_ss_scores.npy` — (N_windows x 234) mapped logits
- `all_ss_embeddings.npy` — (N_windows x 1536) Perch embeddings

In [ ]:
import subprocess, sys
from pathlib import Path

# kernel_sources mount under /kaggle/input/notebooks/ or /kaggle/input/
_WHL_CANDIDATES = [
    Path('/kaggle/input/bc26-tensorflow-2-20-0/wheel'),
    Path('/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel'),
    Path('/kaggle/input/kdmitrie/bc26-tensorflow-2-20-0/wheel'),
]
_WHL = next((p for p in _WHL_CANDIDATES if p.exists()), None)

if _WHL is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        str(_WHL / 'tensorboard-2.20.0-py3-none-any.whl')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        str(_WHL / 'tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl')], check=True)
    print(f'Installed TF 2.20.0 from {_WHL}')
else:
    print('TF 2.20 wheel not found. Checking paths...')
    for p in _WHL_CANDIDATES:
        print(f'  {p} -> exists={p.exists()}')
    import glob as _g
    whl = _g.glob('/kaggle/input/**/tensorflow*.whl', recursive=True)
    if whl:
        print(f'Found wheels: {whl}')
        for w in whl:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', w], check=True)
        print('Installed TF from discovered wheels.')
    else:
        print('No TF wheels found anywhere. Using default TF.')

In [ ]:
import os
import re
import gc
import time
import glob
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm import tqdm

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')

print(f'TensorFlow {tf.__version__}')

In [ ]:
# ── Config ──
COMP_CANDIDATES = [
    Path('/kaggle/input/competitions/birdclef-2026'),
    Path('/kaggle/input/birdclef-2026'),
]
COMP = next((p for p in COMP_CANDIDATES if p.exists()), COMP_CANDIDATES[0])
TRAIN_SS_DIR = COMP / 'train_soundscapes'

MODEL_DIR = Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1')

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC   # 160,000
N_WINDOWS = 12
FILE_SAMPLES = WINDOW_SAMPLES * N_WINDOWS  # 1,920,000 (60s)
BATCH_FILES = 8  # files per batch to manage memory

print(f'COMP       : {COMP}')
print(f'MODEL_DIR  : {MODEL_DIR}')
print(f'TRAIN_SS   : {TRAIN_SS_DIR}')

# ── Load Perch model ──
print('Loading Perch v2 model...')
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures['serving_default']
print('Perch loaded.')

# ── Build species mapping (taxonomy -> Perch 14k classes) ──
bc_labels = pd.read_csv(MODEL_DIR / 'assets' / 'labels.csv').reset_index().rename(
    columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'})
NO_LABEL_INDEX = len(bc_labels)

taxonomy = pd.read_csv(COMP / 'taxonomy.csv')
taxonomy['scientific_name'] = taxonomy['scientific_name'].astype(str)
sample_sub = pd.read_csv(COMP / 'sample_submission.csv')
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

mapping = taxonomy.merge(bc_labels[['scientific_name', 'bc_index']], on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL_INDEX).astype(int)
label_to_bc = mapping.set_index('primary_label')['bc_index']
BC_INDICES = np.array([int(label_to_bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

# Genus-level proxy mapping for unmapped species
unmapped_df = mapping[mapping['bc_index'] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[~unmapped_df['primary_label'].astype(str).str.contains('son', na=False)].copy()
proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    genus = str(row['scientific_name']).split()[0]
    hits = bc_labels[bc_labels['scientific_name'].str.match(rf'^{re.escape(genus)}\s', na=False)]
    if len(hits) > 0:
        proxy_map[str(row['primary_label'])] = hits['bc_index'].astype(int).tolist()

selected_proxy_pos_to_bc = {}
for lbl, bc_idxs in proxy_map.items():
    if lbl in label_to_idx:
        selected_proxy_pos_to_bc[label_to_idx[lbl]] = np.array(bc_idxs, dtype=np.int32)

print(f'Classes    : {N_CLASSES}')
print(f'Mapped     : {MAPPED_MASK.sum()} / {N_CLASSES}')
print(f'Proxy      : {len(selected_proxy_pos_to_bc)}')

In [ ]:
# ── Filename parser ──
FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {'site': None, 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}


def read_soundscape_60s(path):
    """Read ogg file, pad/trim to exactly 60 seconds."""
    y, sr = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    return y[:FILE_SAMPLES]


def infer_perch_batch(paths, verbose=True):
    """Batch process soundscape files through Perch.
    
    Returns:
        meta_df: DataFrame with row_id, filename, site, hour_utc
        scores: (n_rows, N_CLASSES) mapped logits
        embeddings: (n_rows, 1536) Perch embeddings
    """
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS

    # Pre-allocate arrays
    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)
    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_row = 0
    itr = tqdm(range(0, n_files, BATCH_FILES), desc='Perch batch',
               disable=not verbose)

    for start in itr:
        batch = paths[start:start + BATCH_FILES]
        bn = len(batch)
        x = np.empty((bn * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        bstart = write_row

        for bi, path in enumerate(batch):
            audio = read_soundscape_60s(path)
            x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = audio.reshape(
                N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_soundscape_filename(path.name)
            row_ids[write_row:write_row + N_WINDOWS] = [
                f'{path.stem}_{t}' for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS] = meta['site']
            hours[write_row:write_row + N_WINDOWS] = meta['hour_utc']
            write_row += N_WINDOWS

        # Run Perch inference
        out = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = out['label'].numpy().astype(np.float32)
        emb = out['embedding'].numpy().astype(np.float32)

        # Map logits to competition classes
        n_chunk = write_row - bstart
        scores[bstart:write_row, MAPPED_POS] = logits[:n_chunk, MAPPED_BC_INDICES]
        embeddings[bstart:write_row] = emb[:n_chunk]

        # Proxy mapping for unmapped species
        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            scores[bstart:write_row, pos] = logits[:n_chunk, bc_idx_arr].max(axis=1)

        del x, out, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        'row_id': row_ids, 'filename': filenames,
        'site': sites, 'hour_utc': hours
    })
    return meta_df, scores, embeddings


print('Inference functions defined.')

In [ ]:
# ── Process ALL train_soundscapes ──
t0 = time.time()

all_ogg = sorted(glob.glob(str(TRAIN_SS_DIR / '*.ogg')))
print(f'Total soundscape files: {len(all_ogg)}')

if len(all_ogg) == 0:
    raise FileNotFoundError(f'No .ogg files found in {TRAIN_SS_DIR}')

# Process all files in one call (internal batching handles memory)
meta_df, scores_arr, emb_arr = infer_perch_batch(all_ogg, verbose=True)

elapsed = time.time() - t0
print(f'\nProcessed {len(all_ogg)} files in {elapsed:.1f}s ({elapsed/60:.1f} min)')
print(f'  meta    : {meta_df.shape}')
print(f'  scores  : {scores_arr.shape}')
print(f'  embeddings : {emb_arr.shape}')

# ── Save outputs ──
OUT_DIR = Path('/kaggle/working')

meta_df.to_parquet(OUT_DIR / 'all_ss_meta.parquet', index=False)
np.save(OUT_DIR / 'all_ss_scores.npy', scores_arr)
np.save(OUT_DIR / 'all_ss_embeddings.npy', emb_arr)

total_time = time.time() - t0
print(f'\nSaved to {OUT_DIR}/')
print(f'Total time (including save): {total_time:.1f}s ({total_time/60:.1f} min)')

In [ ]:
# ── Verify outputs ──
OUT_DIR = Path('/kaggle/working')

meta_check = pd.read_parquet(OUT_DIR / 'all_ss_meta.parquet')
scores_check = np.load(OUT_DIR / 'all_ss_scores.npy')
emb_check = np.load(OUT_DIR / 'all_ss_embeddings.npy')

print('=== Output Shapes ===')
print(f'  meta       : {meta_check.shape}')
print(f'  scores     : {scores_check.shape}  (dtype={scores_check.dtype})')
print(f'  embeddings : {emb_check.shape}  (dtype={emb_check.dtype})')

print('\n=== Disk Usage ===')
for fname in ['all_ss_meta.parquet', 'all_ss_scores.npy', 'all_ss_embeddings.npy']:
    fpath = OUT_DIR / fname
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f'  {fname:30s} {size_mb:8.1f} MB')

print('\n=== Sample Rows ===')
display(meta_check.head(15))

print('\n=== Score Stats ===')
print(f'  min={scores_check.min():.4f}  max={scores_check.max():.4f}  mean={scores_check.mean():.6f}')
print(f'  nonzero fraction: {(scores_check != 0).mean():.4f}')

print('\n=== Embedding Stats ===')
print(f'  min={emb_check.min():.4f}  max={emb_check.max():.4f}  mean={emb_check.mean():.6f}')
print(f'  L2 norm (first row): {np.linalg.norm(emb_check[0]):.4f}')